<a href="https://colab.research.google.com/github/carlossanzp4-glitch/SanchezParedes-laboratorios/blob/main/Tarea_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
# Tabla 1: Base de Empleados (Con errores intencionales)
df_empleados = pd.DataFrame({
"id_emp": ["E001", " E002 ", "e003", "E004"],
"nombre_completo": [" ana perez", "LUIS GOMEZ", "carlos ruiz", "maria diaz "],
"salario_base": ["USD 1200", "USD 1500", "USD 900", "USD 2000"]
})
# Tabla 2: Evaluaciones de Desempeño
df_evaluaciones = pd.DataFrame({
"id_empleado": ["E001", "E002", "E003", "E009"], # Nota: E009 no existe en la base principal
"calificacion": ["A", "B", "A", "C"]
})

Misiones:
1. Limpie la columna id_emp del df_empleados. Utilice .apply(lambda x: ...) o los métodos
encadenados de .str para eliminar los espacios en blanco a los extremos y convertir
todas las letras a mayúsculas.
2. Estandarice la columna nombre_completo para que no tenga espacios sueltos a los lados
y utilice el formato de Título (Ej. "Ana Perez").
3. Limpie la columna salario_base utilizando .apply() y una función lambda para quitar el
texto "USD " y convertir el valor resultante a un número decimal (float).

In [ ]:
df_empleados['id_emp'] = df_empleados['id_emp'].apply(lambda x : x.strip().upper())
df_empleados['nombre_completo'] = df_empleados['nombre_completo'].str.strip().str.title()
df_empleados['salario_base'] = df_empleados['salario_base'].apply(lambda x : float(x.strip('USD ')))


In [ ]:
df_empleados

,id_emp,nombre_completo,salario_base
0,E001,Ana Perez,1200.0
1,E002,Luis Gomez,1500.0
2,E003,Carlos Ruiz,900.0
3,E004,Maria Diaz,2000.0


Parte 2: Cruce de Tablas (Merge)
Ahora debe unir la información de desempeño a la base principal de empleados.
Misiones:
1. Utilice pd.merge() para realizar un cruce tipo LEFT JOIN donde df_empleados sea la
tabla "mandante" (a la izquierda). Queremos mantener a todos los empleados de la base,
tengan o no evaluación.
2. Dado que las llaves se llaman distinto (id_emp en la primera tabla vs id_empleado en la
segunda), utilice los parámetros left_on y right_on.
3. Guarde la tabla cruzada resultante en una variable llamada df_consolidado.

4. Notará que el cruce generó columnas redundantes. Elimine la columna id_empleado que
provino de la tabla de la derecha utilizando .drop(..., axis=1).

In [ ]:
df_consolidado= pd.merge(df_empleados, df_evaluaciones, left_on='id_emp', right_on='id_empleado', how='left')

In [ ]:
df_consolidado= df_consolidado.drop('id_empleado',axis=1)

Parte 3: Lógica Condicional Multivariable (.apply con def)
Finanzas ha determinado la siguiente regla para asignar los bonos de desempeño a la planilla
cruzada.
Misiones:
1. Cree una función en Python (utilizando def) llamada calcular_bono(fila) que reciba una
fila completa de la tabla.
2. La lógica interna de la función debe ser:
○ Si la calificación es "A", el bono será equivalente al 20% del salario_base.
○ Si la calificación es "B", el bono será equivalente al 10% del salario_base.
○ Si la calificación es nula o cualquier otra letra, el bono será 0.
3. Aplique su función al df_consolidado utilizando el método .apply(..., axis=1) para crear
una nueva columna llamada bono_asignado. (Recuerde que axis=1 es vital para que la
función pueda leer múltiples columnas de la misma fila).
4. Cree una columna final llamada salario_neto que sume el salario base más el bono
asignado.

In [ ]:
def calcular_bono(fila):
  calificacion = fila['calificacion']
  salario = fila['salario_base']
  if calificacion == 'A':
    return round(salario*0.2,2)
  elif calificacion == "B":
    return round(salario*0.1,2)
  else:
    return 0


In [ ]:
df_consolidado["bono"] = df_consolidado.apply(calcular_bono, axis=1)

In [ ]:
df_consolidado

,id_emp,nombre_completo,salario_base,calificacion,bono,salario_neto
0,E001,Ana Perez,1200.0,A,240.0,1440.0
1,E002,Luis Gomez,1500.0,B,150.0,1650.0
2,E003,Carlos Ruiz,900.0,A,180.0,1080.0
3,E004,Maria Diaz,2000.0,NaN,0.0,2000.0


In [ ]:
df_consolidado["salario_neto"] = df_consolidado["bono"]+df_consolidado["salario_base"]

In [ ]:
df_consolidado

,id_emp,nombre_completo,salario_base,calificacion,bono,salario_neto
0,E001,Ana Perez,1200.0,A,240.0,1440.0
1,E002,Luis Gomez,1500.0,B,150.0,1650.0
2,E003,Carlos Ruiz,900.0,A,180.0,1080.0
3,E004,Maria Diaz,2000.0,NaN,0.0,2000.0


Parte 4: Fase LOAD (Exportaciones al Sistema)
El pipeline ETL no termina hasta que los datos no son cargados a su destino final.
Misiones:
1. Exporte el DataFrame final df_consolidado a un archivo plano llamado
"planilla_final_2026.csv".
2. Asegúrese de incluir el parámetro index=False para que el sistema financiero no reciba
una columna extra de números de índice sin sentido.
3. (Reto Adicional) Exporte el mismo DataFrame a un archivo de Excel (.xlsx) en una
pestaña que se llame "Planilla_Neta".

In [ ]:
# 1. Exportar a CSV
df_consolidado.to_csv('planilla_final_2026.csv', index=False)
print('DataFrame exportado a planilla_final_2026.csv')


DataFrame exportado a planilla_final_2026.csv


In [ ]:
# 3. Exportar a Excel
# Se necesita la librería openpyxl para exportar a Excel
# Si no la tienes instalada, puedes instalarla con: !pip install openpyxl

with pd.ExcelWriter('planilla_final_2026.xlsx', engine='openpyxl') as writer:
    df_consolidado.to_excel(writer, sheet_name='Planilla_Neta', index=False)
print('DataFrame exportado a planilla_final_2026.xlsx en la pestaña Planilla_Neta')


DataFrame exportado a planilla_final_2026.xlsx en la pestaña Planilla_Neta
